# **Tahap Pelatihan & Evaluasi Model Binary**

## **Langkah 9: Model Training - Binary OvR (One vs Rest)**


####**9.1 Setup dan Definisi Fungsi**

In [ ]:
label_list = [LABEL_NAMES_IIOT[i] for i in range(N_CLASSES)]

# Mapping model -> direktori simpan
MODEL_DIRS = {
    'RandomForest': BINARY_RF_DIR_IIOT,
    'XGBoost'     : BINARY_XGB_DIR_IIOT,
    'LightGBM'    : BINARY_LGB_DIR_IIOT,
    'DNN'         : BINARY_DNN_DIR_IIOT,
}

print(f"Strategi          : One vs Rest (OvR)")
print(f"Jumlah kelas      : {N_CLASSES}")
print(f"Jumlah classifier : 4 (RF, XGB, LGB, DNN)")
print(f"Total model       : {N_CLASSES * 4} model")
print(f"\nKelas yang dilatih:")
for enc, name in LABEL_NAMES_IIOT.items():
    n_pos = int((y_train_final == enc).sum())
    n_neg = int((y_train_final != enc).sum())
    ratio = n_neg / n_pos if n_pos > 0 else 0
    print(f"  {enc} {name:<18} : pos={n_pos:>10,} | neg={n_neg:>10,} | ratio=1:{ratio:.1f}")

mem = psutil.virtual_memory()
print(f"\nRAM tersisa: {mem.available / 1024**3:.2f} GB")

Strategi          : One vs Rest (OvR)
Jumlah kelas      : 8
Jumlah classifier : 4 (RF, XGB, LGB, DNN)
Total model       : 32 model

Kelas yang dilatih:
  0 Benign             : pos=   163,578 | neg= 1,057,535 | ratio=1:6.5
  1 Brute_Force        : pos=   163,578 | neg= 1,057,535 | ratio=1:6.5
  2 DDoS               : pos=   163,578 | neg= 1,057,535 | ratio=1:6.5
  3 DoS                : pos=   163,578 | neg= 1,057,535 | ratio=1:6.5
  4 MITM/Spoofing      : pos=   163,578 | neg= 1,057,535 | ratio=1:6.5
  5 Malware/Mirai      : pos=   163,578 | neg= 1,057,535 | ratio=1:6.5
  6 Recon              : pos=    76,067 | neg= 1,145,046 | ratio=1:15.1
  7 Web_Based          : pos=   163,578 | neg= 1,057,535 | ratio=1:6.5

RAM tersisa: 47.70 GB


####**9.2 Definisi Fungsi Evaluasi Binary**

In [ ]:
print("Mendefinisikan fungsi evaluasi binary")

from sklearn.metrics import roc_auc_score

def evaluate_binary(model_name, class_name, y_true_bin, y_pred_bin, y_prob_pos=None):
    acc    = accuracy_score(y_true_bin, y_pred_bin)
    prec   = precision_score(y_true_bin, y_pred_bin, zero_division=0)
    rec    = recall_score(y_true_bin, y_pred_bin, zero_division=0)
    f1     = f1_score(y_true_bin, y_pred_bin, zero_division=0)
    mcc    = matthews_corrcoef(y_true_bin, y_pred_bin)
    roc_auc = None
    if y_prob_pos is not None:
        try:
            roc_auc = roc_auc_score(y_true_bin, y_prob_pos)
        except Exception:
            roc_auc = None

    return {
        'model'     : model_name,
        'class'     : class_name,
        'accuracy'  : round(float(acc),  6),
        'precision' : round(float(prec), 6),
        'recall'    : round(float(rec),  6),
        'f1'        : round(float(f1),   6),
        'mcc'       : round(float(mcc),  6),
        'roc_auc'   : round(float(roc_auc), 6) if roc_auc is not None else None,
        'support_pos': int(y_true_bin.sum()),
        'support_neg': int((y_true_bin == 0).sum()),
    }

def print_binary_results(model_name, class_name, results):
    print(f"\n{model_name} [{class_name}] - Hasil Evaluasi:")
    print(f"  Accuracy   : {results['accuracy']:.6f}")
    print(f"  Precision  : {results['precision']:.6f}")
    print(f"  Recall     : {results['recall']:.6f}")
    print(f"  F1-Score   : {results['f1']:.6f}")
    print(f"  MCC        : {results['mcc']:.6f}")
    if results['roc_auc'] is not None:
        print(f"  ROC-AUC    : {results['roc_auc']:.6f}")
    print(f"  Support    : pos={results['support_pos']:,} | neg={results['support_neg']:,}")

def save_binary_cm(y_true_bin, y_pred_bin, model_name, class_name, save_dir):
    cm   = confusion_matrix(y_true_bin, y_pred_bin)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=[f'Bukan {class_name}', class_name],
                yticklabels=[f'Bukan {class_name}', class_name], ax=ax)
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('True',      fontsize=10)
    ax.set_title(f'CM - {model_name} [{class_name}]\nBinary OvR, CIC IIoT 2025',
                 fontsize=10)
    plt.tight_layout()
    safe_name = class_name.replace('/', '_').replace(' ', '_')
    path      = os.path.join(save_dir, f'CM_{model_name}_{safe_name}_binary.png')
    plt.savefig(path, dpi=120, bbox_inches='tight')
    plt.close()
    return path

def build_binary_dnn(input_dim, learning_rate=0.001,
                     hidden_layers=[128, 64, 32, 16], dropout_rate=0.3):
    inp = keras.Input(shape=(input_dim,), name='input')
    x   = inp
    for i, units in enumerate(hidden_layers):
        x = layers.Dense(units, name=f'dense_{i+1}')(x)
        x = layers.BatchNormalization(name=f'bn_{i+1}')(x)
        x = layers.Activation('relu', name=f'act_{i+1}')(x)
        x = layers.Dropout(dropout_rate, name=f'dropout_{i+1}')(x)
    out   = layers.Dense(1, activation='sigmoid', name='output')(x)
    model = keras.Model(inp, out, name='DNN_binary')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

print("  evaluate_binary      : OK")
print("  print_binary_results : OK")
print("  save_binary_cm       : OK")
print("  build_binary_dnn     : OK")

# Storage hasil semua model semua kelas
binary_results    = {m: {} for m in ['RandomForest', 'XGBoost', 'LightGBM', 'DNN']}
binary_train_time = {m: {} for m in ['RandomForest', 'XGBoost', 'LightGBM', 'DNN']}

Mendefinisikan fungsi evaluasi binary
  evaluate_binary      : OK
  print_binary_results : OK
  save_binary_cm       : OK
  build_binary_dnn     : OK


####**9.3 Random Forest - Binary OvR**

In [ ]:
RF_PARAMS_BIN = {
    'n_estimators'     : 100,
    'max_depth'        : 15,
    'min_samples_split': 5,
    'min_samples_leaf' : 2,
    'max_features'     : 'sqrt',
    'n_jobs'           : -1,
    'random_state'     : SEED,
    'verbose'          : 0,
}

print(f"Parameter RF Binary:")
for k, v in RF_PARAMS_BIN.items():
    print(f"  {k:<22} : {v}")

for enc in range(N_CLASSES):
    class_name = LABEL_NAMES_IIOT[enc]
    print(f"\n--- [{enc+1}/{N_CLASSES}] {class_name} ---")

    # Buat label binary
    y_train_bin = (y_train_final == enc).astype(np.int32)
    y_test_bin  = (y_test        == enc).astype(np.int32)

    # Training
    t0       = time.time()
    rf_bin   = RandomForestClassifier(**RF_PARAMS_BIN)
    rf_bin.fit(X_train_final, y_train_bin)
    t_train  = time.time() - t0

    # Prediksi
    print(f"Prediksi pada test set...")
    y_pred_bin = rf_bin.predict(X_test)
    y_prob_bin = rf_bin.predict_proba(X_test)[:, 1]

    # Evaluasi
    res = evaluate_binary('RandomForest', class_name, y_test_bin, y_pred_bin, y_prob_bin)
    res['train_time_s'] = round(t_train, 2)
    print_binary_results('RandomForest', class_name, res)

    # Confusion matrix
    cm_path = save_binary_cm(
        y_test_bin, y_pred_bin, 'RandomForest', class_name, BINARY_CM_DIR_IIOT
    )
    print(f"  Confusion matrix tersimpan: {cm_path}")

    # Simpan model
    safe_name = class_name.replace('/', '_').replace(' ', '_')
    model_path = os.path.join(BINARY_RF_DIR_IIOT, f'RF_binary_{safe_name}.pkl')
    joblib.dump(rf_bin, model_path)
    print(f"  Model tersimpan: {model_path}")

    binary_results['RandomForest'][class_name]    = res
    binary_train_time['RandomForest'][class_name] = t_train

    del rf_bin, y_train_bin, y_pred_bin, y_prob_bin
    gc.collect()

print(f"\nRandomForest Binary OvR selesai.")
mem = psutil.virtual_memory()
print(f"RAM tersisa: {mem.available / 1024**3:.2f} GB")

# Ringkasan OvR - RandomForest
rf_res = binary_results['RandomForest']
macro_prec = np.mean([rf_res[c]['precision'] for c in label_list])
macro_rec  = np.mean([rf_res[c]['recall'] for c in label_list])
macro_f1   = np.mean([rf_res[c]['f1'] for c in label_list])
macro_mcc  = np.mean([rf_res[c]['mcc'] for c in label_list])
macro_roc  = np.mean([rf_res[c]['roc_auc'] for c in label_list if rf_res[c].get('roc_auc') is not None])

print("\n  " + "=" * 50)
print("  Ringkasan OvR - RandomForest")
print("  " + "=" * 50)
print(f"  Macro Precision : {macro_prec:.6f}")
print(f"  Macro Recall    : {macro_rec:.6f}")
print(f"  Macro F1        : {macro_f1:.6f}")
print(f"  Macro MCC       : {macro_mcc:.6f}")
print(f"  Macro ROC-AUC   : {macro_roc:.6f}")

Parameter RF Binary:
  n_estimators           : 100
  max_depth              : 15
  min_samples_split      : 5
  min_samples_leaf       : 2
  max_features           : sqrt
  n_jobs                 : -1
  random_state           : 42
  verbose                : 0

--- [1/8] Benign ---
Prediksi pada test set...

RandomForest [Benign] - Hasil Evaluasi:
  Accuracy   : 0.995543
  Precision  : 0.987753
  Recall     : 0.999555
  F1-Score   : 0.993619
  MCC        : 0.990236
  ROC-AUC    : 0.999734
  Support    : pos=15,734 | neg=29,585
  Confusion matrix tersimpan: /content/drive/My Drive/Framework/CICIIoT2025/Binary/Results/Confusion_Matrices/CM_RandomForest_Benign_binary.png
  Model tersimpan: /content/drive/My Drive/Framework/CICIIoT2025/Binary/Models/RandomForest/RF_binary_Benign.pkl

--- [2/8] Brute_Force ---
Prediksi pada test set...

RandomForest [Brute_Force] - Hasil Evaluasi:
  Accuracy   : 0.999956
  Precision  : 1.000000
  Recall     : 0.996497
  F1-Score   : 0.998246
  MCC        : 

####**9.4 XGBoost - Binary OvR**

In [ ]:
XGB_PARAMS_BIN = {
    'n_estimators'     : 100,
    'max_depth'        : 8,
    'learning_rate'    : 0.1,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'objective'        : 'binary:logistic',
    'eval_metric'      : 'logloss',
    'use_label_encoder': False,
    'tree_method'      : 'hist',
    'device'           : 'cuda',
    'random_state'     : SEED,
    'n_jobs'           : -1,
}

print(f"Parameter XGBoost Binary:")
for k, v in XGB_PARAMS_BIN.items():
    print(f"  {k:<22} : {v}")

try:
    xgb_test = xgb.XGBClassifier(device='cuda', n_estimators=1)
    xgb_test.fit(X_train_final[:10], (y_train_final[:10] == 0).astype(int))
    print(f"\nXGBoost GPU (cuda) tersedia.")
except Exception:
    XGB_PARAMS_BIN['device'] = 'cpu'
    print(f"\nXGBoost GPU tidak tersedia, menggunakan CPU.")

for enc in range(N_CLASSES):
    class_name = LABEL_NAMES_IIOT[enc]
    print(f"\n--- [{enc+1}/{N_CLASSES}] {class_name} ---")

    y_train_bin = (y_train_final == enc).astype(np.int32)
    y_test_bin  = (y_test        == enc).astype(np.int32)

    t0      = time.time()
    xgb_bin = xgb.XGBClassifier(**XGB_PARAMS_BIN)
    xgb_bin.fit(
        X_train_final, y_train_bin,
        eval_set=[(X_test, y_test_bin)],
        verbose=50
    )
    t_train = time.time() - t0

    print(f"Prediksi pada test set...")
    y_pred_bin = xgb_bin.predict(X_test)
    y_prob_bin = xgb_bin.predict_proba(X_test)[:, 1]

    res = evaluate_binary('XGBoost', class_name, y_test_bin, y_pred_bin, y_prob_bin)
    res['train_time_s'] = round(t_train, 2)
    print_binary_results('XGBoost', class_name, res)

    cm_path = save_binary_cm(
        y_test_bin, y_pred_bin, 'XGBoost', class_name, BINARY_CM_DIR_IIOT
    )
    print(f"  Confusion matrix tersimpan: {cm_path}")

    safe_name  = class_name.replace('/', '_').replace(' ', '_')
    model_path = os.path.join(BINARY_XGB_DIR_IIOT, f'XGB_binary_{safe_name}.json')
    xgb_bin.save_model(model_path)
    print(f"  Model tersimpan: {model_path}")

    binary_results['XGBoost'][class_name]    = res
    binary_train_time['XGBoost'][class_name] = t_train

    del xgb_bin, y_train_bin, y_pred_bin, y_prob_bin
    gc.collect()

print(f"\nXGBoost Binary OvR selesai.")
mem = psutil.virtual_memory()
print(f"RAM tersisa: {mem.available / 1024**3:.2f} GB")

# Ringkasan OvR - XGBoost
xgb_res = binary_results['XGBoost']
macro_prec = np.mean([xgb_res[c]['precision'] for c in label_list])
macro_rec  = np.mean([xgb_res[c]['recall'] for c in label_list])
macro_f1   = np.mean([xgb_res[c]['f1'] for c in label_list])
macro_mcc  = np.mean([xgb_res[c]['mcc'] for c in label_list])
macro_roc  = np.mean([xgb_res[c]['roc_auc'] for c in label_list if xgb_res[c].get('roc_auc') is not None])

print("\n  " + "=" * 50)
print("  Ringkasan OvR - XGBoost")
print("  " + "=" * 50)
print(f"  Macro Precision : {macro_prec:.6f}")
print(f"  Macro Recall    : {macro_rec:.6f}")
print(f"  Macro F1        : {macro_f1:.6f}")
print(f"  Macro MCC       : {macro_mcc:.6f}")
print(f"  Macro ROC-AUC   : {macro_roc:.6f}")

Parameter XGBoost Binary:
  n_estimators           : 100
  max_depth              : 8
  learning_rate          : 0.1
  subsample              : 0.8
  colsample_bytree       : 0.8
  objective              : binary:logistic
  eval_metric            : logloss
  use_label_encoder      : False
  tree_method            : hist
  device                 : cuda
  random_state           : 42
  n_jobs                 : -1

XGBoost GPU (cuda) tersedia.

--- [1/8] Benign ---
[0]	validation_0-logloss:0.58061
[50]	validation_0-logloss:0.01737
[99]	validation_0-logloss:0.01219
Prediksi pada test set...

XGBoost [Benign] - Hasil Evaluasi:
  Accuracy   : 0.996447
  Precision  : 0.990674
  Recall     : 0.999174
  F1-Score   : 0.994906
  MCC        : 0.992200
  ROC-AUC    : 0.999850
  Support    : pos=15,734 | neg=29,585
  Confusion matrix tersimpan: /content/drive/My Drive/Framework/CICIIoT2025/Binary/Results/Confusion_Matrices/CM_XGBoost_Benign_binary.png
  Model tersimpan: /content/drive/My Drive/Framew

####**9.5 LightGBM - Binary OvR**

In [ ]:
LGB_PARAMS_BIN = {
    'n_estimators'     : 100,
    'max_depth'        : 8,
    'learning_rate'    : 0.1,
    'num_leaves'       : 31,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'objective'        : 'binary',
    'metric'           : 'binary_logloss',
    'device'           : 'cpu',
    'random_state'     : SEED,
    'n_jobs'           : -1,
    'verbose'          : 1,
}

print(f"Parameter LightGBM Binary:")
for k, v in LGB_PARAMS_BIN.items():
    print(f"  {k:<22} : {v}")

try:
    lgb_test = lgb.LGBMClassifier(device='gpu', n_estimators=1, verbose=-1)
    lgb_test.fit(X_train_final[:10], (y_train_final[:10] == 0).astype(int))
    print(f"\nLightGBM GPU tersedia.")
except Exception:
    LGB_PARAMS_BIN['device'] = 'cpu'
    print(f"\nLightGBM GPU tidak tersedia, menggunakan CPU.")

for enc in range(N_CLASSES):
    class_name = LABEL_NAMES_IIOT[enc]
    print(f"\n--- [{enc+1}/{N_CLASSES}] {class_name} ---")

    y_train_bin = (y_train_final == enc).astype(np.int32)
    y_test_bin  = (y_test        == enc).astype(np.int32)

    t0      = time.time()
    lgb_bin = lgb.LGBMClassifier(**LGB_PARAMS_BIN)
    lgb_bin.fit(
        X_train_final, y_train_bin,
        eval_set=[(X_test, y_test_bin)],
        callbacks=[lgb.log_evaluation(period=50)]
    )
    t_train = time.time() - t0

    print(f"Prediksi pada test set...")
    y_pred_bin = lgb_bin.predict(X_test)
    y_prob_bin = lgb_bin.predict_proba(X_test)[:, 1]

    res = evaluate_binary('LightGBM', class_name, y_test_bin, y_pred_bin, y_prob_bin)
    res['train_time_s'] = round(t_train, 2)
    print_binary_results('LightGBM', class_name, res)

    cm_path = save_binary_cm(
        y_test_bin, y_pred_bin, 'LightGBM', class_name, BINARY_CM_DIR_IIOT
    )
    print(f"  Confusion matrix tersimpan: {cm_path}")

    safe_name  = class_name.replace('/', '_').replace(' ', '_')
    model_path = os.path.join(BINARY_LGB_DIR_IIOT, f'LGB_binary_{safe_name}.pkl')
    joblib.dump(lgb_bin, model_path)
    print(f"  Model tersimpan: {model_path}")

    binary_results['LightGBM'][class_name]    = res
    binary_train_time['LightGBM'][class_name] = t_train

    del lgb_bin, y_train_bin, y_pred_bin, y_prob_bin
    gc.collect()

print(f"\nLightGBM Binary OvR selesai.")
mem = psutil.virtual_memory()
print(f"RAM tersisa: {mem.available / 1024**3:.2f} GB")

# Ringkasan OvR - LightGBM
lgb_res = binary_results['LightGBM']
macro_prec = np.mean([lgb_res[c]['precision'] for c in label_list])
macro_rec  = np.mean([lgb_res[c]['recall'] for c in label_list])
macro_f1   = np.mean([lgb_res[c]['f1'] for c in label_list])
macro_mcc  = np.mean([lgb_res[c]['mcc'] for c in label_list])
macro_roc  = np.mean([lgb_res[c]['roc_auc'] for c in label_list if lgb_res[c].get('roc_auc') is not None])

print("\n  " + "=" * 50)
print("  Ringkasan OvR - LightGBM")
print("  " + "=" * 50)
print(f"  Macro Precision : {macro_prec:.6f}")
print(f"  Macro Recall    : {macro_rec:.6f}")
print(f"  Macro F1        : {macro_f1:.6f}")
print(f"  Macro MCC       : {macro_mcc:.6f}")
print(f"  Macro ROC-AUC   : {macro_roc:.6f}")

Parameter LightGBM Binary:
  n_estimators           : 100
  max_depth              : 8
  learning_rate          : 0.1
  num_leaves             : 31
  subsample              : 0.8
  colsample_bytree       : 0.8
  objective              : binary
  metric                 : binary_logloss
  device                 : cpu
  random_state           : 42
  n_jobs                 : -1
  verbose                : 1

LightGBM GPU tersedia.

--- [1/8] Benign ---
[LightGBM] [Info] Number of positive: 163578, number of negative: 1057535
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.123085 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15085
[LightGBM] [Info] Number of data points in the train set: 1221113, number of used features: 71
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.133958 -> initscore=-1.866406
[LightGBM] [Info] Start training from 

####**9.6 DNN - Binary OvR**

In [ ]:
DNN_PARAMS_BIN = {
    'hidden_layers' : [128, 64, 32, 16],
    'dropout_rate'  : 0.3,
    'learning_rate' : 0.001,
    'batch_size'    : 256,
    'epochs'        : 50,
    'patience'      : 5,
}

print(f"Parameter DNN Binary:")
for k, v in DNN_PARAMS_BIN.items():
    print(f"  {k:<18} : {v}")

for enc in range(N_CLASSES):
    class_name = LABEL_NAMES_IIOT[enc]
    safe_name  = class_name.replace('/', '_').replace(' ', '_')
    print(f"\n{'='*55}")
    print(f"[{enc+1}/{N_CLASSES}] DNN - {class_name}")
    print(f"{'='*55}")

    y_train_bin = (y_train_final == enc).astype(np.float32)
    y_test_bin  = (y_test        == enc).astype(np.int32)

    n_pos = int(y_train_bin.sum())
    n_neg = int((y_train_bin == 0).sum())
    print(f"  Train: pos={n_pos:,} | neg={n_neg:,}")

    tf.keras.backend.clear_session()
    dnn_bin = build_binary_dnn(
        input_dim    = N_FEATURES,
        learning_rate= DNN_PARAMS_BIN['learning_rate'],
        hidden_layers= DNN_PARAMS_BIN['hidden_layers'],
        dropout_rate = DNN_PARAMS_BIN['dropout_rate'],
    )

    best_path = os.path.join(BINARY_DNN_DIR_IIOT,
                             f'DNN_binary_{safe_name}_best.keras')
    callbacks_bin = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=DNN_PARAMS_BIN['patience'],
            restore_best_weights=True, verbose=0
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3,
            min_lr=1e-5, verbose=0
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=best_path, monitor='val_loss',
            save_best_only=True, verbose=0
        ),
    ]

    print(f"  Memulai training DNN...")
    t0          = time.time()
    dnn_history = dnn_bin.fit(
        X_train_final, y_train_bin,
        epochs           = DNN_PARAMS_BIN['epochs'],
        batch_size       = DNN_PARAMS_BIN['batch_size'],
        validation_split = 0.1,
        callbacks        = callbacks_bin,
        verbose          = 0,
        shuffle          = True,
    )
    t_train      = time.time() - t0
    n_epochs_run = len(dnn_history.history['loss'])

    print(f"  Training selesai dalam : {t_train:.1f} detik")
    print(f"  Epochs dijalankan      : {n_epochs_run}/{DNN_PARAMS_BIN['epochs']}")
    print(f"  Final train loss       : {dnn_history.history['loss'][-1]:.4f}")
    print(f"  Final val loss         : {dnn_history.history['val_loss'][-1]:.4f}")

    print(f"  Prediksi pada test set...")
    y_prob_bin  = dnn_bin.predict(X_test, batch_size=1024, verbose=0).flatten()
    y_pred_bin  = (y_prob_bin >= 0.5).astype(np.int32)

    res = evaluate_binary('DNN', class_name, y_test_bin, y_pred_bin, y_prob_bin)
    res['train_time_s'] = round(t_train, 2)
    res['n_epochs']     = n_epochs_run
    print_binary_results('DNN', class_name, res)

    cm_path = save_binary_cm(
        y_test_bin, y_pred_bin, 'DNN', class_name, BINARY_CM_DIR_IIOT
    )
    print(f"  Confusion matrix tersimpan: {cm_path}")

    model_path = os.path.join(BINARY_DNN_DIR_IIOT, f'DNN_binary_{safe_name}.keras')
    dnn_bin.save(model_path)

    hist_path = os.path.join(BINARY_DNN_DIR_IIOT,
                             f'DNN_binary_{safe_name}_history.json')
    with open(hist_path, 'w') as f:
        json.dump({
            'loss'        : [float(v) for v in dnn_history.history['loss']],
            'val_loss'    : [float(v) for v in dnn_history.history['val_loss']],
            'accuracy'    : [float(v) for v in dnn_history.history['accuracy']],
            'val_accuracy': [float(v) for v in dnn_history.history['val_accuracy']],
            'n_epochs'    : n_epochs_run,
        }, f, indent=4)

    print(f"  Model tersimpan: {model_path}")

    binary_results['DNN'][class_name]    = res
    binary_train_time['DNN'][class_name] = t_train

    del dnn_bin, y_train_bin, y_pred_bin, y_prob_bin
    tf.keras.backend.clear_session()
    gc.collect()

    mem = psutil.virtual_memory()
    print(f"  RAM tersisa: {mem.available / 1024**3:.2f} GB")

print(f"\nDNN Binary OvR selesai.")

# Ringkasan OvR - DNN
dnn_res = binary_results['DNN']
macro_prec = np.mean([dnn_res[c]['precision'] for c in label_list])
macro_rec  = np.mean([dnn_res[c]['recall'] for c in label_list])
macro_f1   = np.mean([dnn_res[c]['f1'] for c in label_list])
macro_mcc  = np.mean([dnn_res[c]['mcc'] for c in label_list])
macro_roc  = np.mean([dnn_res[c]['roc_auc'] for c in label_list if dnn_res[c].get('roc_auc') is not None])

print("\n  " + "=" * 50)
print("  Ringkasan OvR - DNN")
print("  " + "=" * 50)
print(f"  Macro Precision : {macro_prec:.6f}")
print(f"  Macro Recall    : {macro_rec:.6f}")
print(f"  Macro F1        : {macro_f1:.6f}")
print(f"  Macro MCC       : {macro_mcc:.6f}")
print(f"  Macro ROC-AUC   : {macro_roc:.6f}")

Parameter DNN Binary:
  hidden_layers      : [128, 64, 32, 16]
  dropout_rate       : 0.3
  learning_rate      : 0.001
  batch_size         : 256
  epochs             : 50
  patience           : 5

[1/8] DNN - Benign
  Train: pos=163,578 | neg=1,057,535
  Memulai training DNN...
  Training selesai dalam : 698.1 detik
  Epochs dijalankan      : 50/50
  Final train loss       : 0.0163
  Final val loss         : 0.0155
  Prediksi pada test set...

DNN [Benign] - Hasil Evaluasi:
  Accuracy   : 0.993204
  Precision  : 0.983574
  Recall     : 0.997076
  F1-Score   : 0.990279
  MCC        : 0.985109
  ROC-AUC    : 0.999512
  Support    : pos=15,734 | neg=29,585
  Confusion matrix tersimpan: /content/drive/My Drive/Framework/CICIIoT2025/Binary/Results/Confusion_Matrices/CM_DNN_Benign_binary.png
  Model tersimpan: /content/drive/My Drive/Framework/CICIIoT2025/Binary/Models/DNN/DNN_binary_Benign.keras
  RAM tersisa: 46.68 GB

[2/8] DNN - Brute_Force
  Train: pos=163,578 | neg=1,057,535
  Memulai

####**9.7 Simpan Semua Hasil Binary**

In [ ]:
binary_json_path = os.path.join(BINARY_RESULTS_DIR_IIOT,
                                'langkah9_binary_results.json')
with open(binary_json_path, 'w') as f:
    json.dump(binary_results, f, indent=4)
print(f"Results JSON tersimpan: {binary_json_path}")

# Simpan summary CSV
rows = []
for model_name in ['RandomForest', 'XGBoost', 'LightGBM', 'DNN']:
    for class_name in label_list:
        res = binary_results[model_name][class_name]
        rows.append({
            'Model'      : model_name,
            'Kelas'      : class_name,
            'Accuracy'   : res['accuracy'],
            'Precision'  : res['precision'],
            'Recall'     : res['recall'],
            'F1'         : res['f1'],
            'MCC'        : res['mcc'],
            'ROC-AUC'    : res.get('roc_auc', None),
            'Train Time(s)': res.get('train_time_s', None),
            'Support Pos': res['support_pos'],
            'Support Neg': res['support_neg'],
        })

df_binary = pd.DataFrame(rows)
binary_csv_path = os.path.join(BINARY_RESULTS_DIR_IIOT,
                               'langkah9_binary_summary.csv')
df_binary.to_csv(binary_csv_path, index=False)
print(f"Results CSV tersimpan : {binary_csv_path}")

Results JSON tersimpan: /content/drive/My Drive/Framework/CICIIoT2025/Binary/Results/langkah9_binary_results.json
Results CSV tersimpan : /content/drive/My Drive/Framework/CICIIoT2025/Binary/Results/langkah9_binary_summary.csv


####**9.8 Ringkasan Langkah 9 Binary**

In [ ]:
model_names_bin = ['RandomForest', 'XGBoost', 'LightGBM', 'DNN']

print(f"\n{'Kelas':<20} {'RF F1':>8} {'XGB F1':>8} "
      f"{'LGB F1':>8} {'DNN F1':>8} {'Avg F1':>8}")
print("-" * 60)
for class_name in label_list:
    f1_vals = [binary_results[m][class_name]['f1'] for m in model_names_bin]
    avg_f1  = np.mean(f1_vals)
    print(f"  {class_name:<18} " +
          " ".join([f"{v:>8.4f}" for v in f1_vals]) +
          f" {avg_f1:>8.4f}")
print("-" * 60)

# Rata-rata F1 per model
print(f"\n{'Model':<14}", end="")
for m in model_names_bin:
    avg = np.mean([binary_results[m][c]['f1'] for c in label_list])
    print(f" {m}: {avg:.4f}", end="  ")
print()

mem = psutil.virtual_memory()
print(f"\nRAM tersisa: {mem.available / 1024**3:.2f} GB")
gc.collect()

print("\nLanjut ke Langkah 10 Binary: Evaluation & Comparison")
print("=" * 75)


Kelas                   RF F1   XGB F1   LGB F1   DNN F1   Avg F1
------------------------------------------------------------
  Benign               0.9936   0.9949   0.9951   0.9903   0.9935
  Brute_Force          0.9982   0.9991   0.9974   0.9082   0.9757
  DDoS                 0.9897   0.9908   0.9872   0.9793   0.9868
  DoS                  0.9920   0.9955   0.9921   0.9712   0.9877
  MITM/Spoofing        0.9881   0.9953   0.9927   0.9475   0.9809
  Malware/Mirai        0.9904   0.9938   0.9917   0.9727   0.9871
  Recon                0.9881   0.9918   0.9897   0.9820   0.9879
  Web_Based            0.9984   0.9984   0.9984   0.9968   0.9980
------------------------------------------------------------

Model          RandomForest: 0.9923   XGBoost: 0.9950   LightGBM: 0.9930   DNN: 0.9685  

RAM tersisa: 45.59 GB

Lanjut ke Langkah 10 Binary: Evaluation & Comparison


##**Langkah 10: Evaluation & Comparison Binary OvR**

####**10.1 Setup dan Load Hasil**

In [ ]:
if 'binary_results' not in globals() or len(binary_results) == 0:
    print("binary_results tidak ditemukan di memory, loading dari disk...")
    binary_json_path = os.path.join(BINARY_RESULTS_DIR_IIOT,
                                    'langkah9_binary_results.json')
    assert os.path.exists(binary_json_path), \
        f"ERROR: {binary_json_path} tidak ditemukan."
    with open(binary_json_path, 'r') as f:
        binary_results = json.load(f)
    print(f"  Loaded: {list(binary_results.keys())}")
else:
    print(f"  binary_results tersedia di memory.")

model_names_bin = ['RandomForest', 'XGBoost', 'LightGBM', 'DNN']
label_list      = [LABEL_NAMES_IIOT[i] for i in range(N_CLASSES)]

# Load baseline binary results jika ada
BASELINE_BINARY_FILE = os.path.join(BASELINE_RESULTS_DIR_IIOT,
                                    'baseline_binary_results.json')
has_baseline_bin = os.path.exists(BASELINE_BINARY_FILE)
if has_baseline_bin:
    with open(BASELINE_BINARY_FILE, 'r') as f:
        baseline_binary = json.load(f)
    print(f"\nBaseline binary results ditemukan.")
else:
    print(f"\nBaseline binary results tidak ditemukan.")
    print(f"  Evaluasi fokus pada perbandingan antar model framework.")

  binary_results tersedia di memory.

Baseline binary results tidak ditemukan.
  Evaluasi fokus pada perbandingan antar model framework.


####**10.2  Tabel Ringkasan F1 per Kelas per Model**

In [ ]:
metrics_bin = [
    ('f1',        'F1-Score'),
    ('precision', 'Precision'),
    ('recall',    'Recall'),
    ('mcc',       'MCC'),
    ('roc_auc',   'ROC-AUC'),
]

for metric_key, metric_label in metrics_bin:
    print(f"\n  [{metric_label}]")
    print(f"  {'Kelas':<20} {'RF':>10} {'XGB':>10} {'LGB':>10} {'DNN':>10} {'Avg':>10}")
    print(f"  {'-'*65}")
    metric_avgs = {m: [] for m in model_names_bin}
    for class_name in label_list:
        row  = f"  {class_name:<20}"
        vals = []
        for m in model_names_bin:
            val = binary_results[m][class_name].get(metric_key, None)
            if val is not None:
                row += f" {val:>10.4f}"
                vals.append(val)
                metric_avgs[m].append(val)
            else:
                row += f" {'N/A':>10}"
        avg = np.mean(vals) if vals else 0
        row += f" {avg:>10.4f}"
        print(row)
    print(f"  {'-'*65}")
    row_avg = f"  {'Macro Avg':<20}"
    for m in model_names_bin:
        avg_m = np.mean(metric_avgs[m]) if metric_avgs[m] else 0
        row_avg += f" {avg_m:>10.4f}"
    print(row_avg)


  [F1-Score]
  Kelas                        RF        XGB        LGB        DNN        Avg
  -----------------------------------------------------------------
  Benign                   0.9936     0.9949     0.9951     0.9903     0.9935
  Brute_Force              0.9982     0.9991     0.9974     0.9082     0.9757
  DDoS                     0.9897     0.9908     0.9872     0.9793     0.9868
  DoS                      0.9920     0.9955     0.9921     0.9712     0.9877
  MITM/Spoofing            0.9881     0.9953     0.9927     0.9475     0.9809
  Malware/Mirai            0.9904     0.9938     0.9917     0.9727     0.9871
  Recon                    0.9881     0.9918     0.9897     0.9820     0.9879
  Web_Based                0.9984     0.9984     0.9984     0.9968     0.9980
  -----------------------------------------------------------------
  Macro Avg                0.9923     0.9950     0.9930     0.9685

  [Precision]
  Kelas                        RF        XGB        LGB        DNN

####**10.3 Tabel Detail per Model**

In [ ]:
for model_name in model_names_bin:
    print(f"\n  [{model_name}]")
    print(f"  {'Kelas':<20} {'Precision':>10} {'Recall':>10} "
          f"{'F1':>8} {'MCC':>8} {'ROC-AUC':>9} {'Support+':>9}")
    print(f"  {'-'*75}")
    f1_list  = []
    mcc_list = []
    for class_name in label_list:
        res  = binary_results[model_name][class_name]
        roc  = res.get('roc_auc', None)
        roc_str = f"{roc:>9.4f}" if roc is not None else f"{'N/A':>9}"
        print(f"  {class_name:<20} {res['precision']:>10.4f} "
              f"{res['recall']:>10.4f} {res['f1']:>8.4f} "
              f"{res['mcc']:>8.4f} {roc_str} {res['support_pos']:>9,}")
        f1_list.append(res['f1'])
        mcc_list.append(res['mcc'])
    print(f"  {'-'*75}")
    print(f"  {'Macro Avg':<20} {'':>10} {'':>10} "
          f"{np.mean(f1_list):>8.4f} {np.mean(mcc_list):>8.4f}")


  [RandomForest]
  Kelas                 Precision     Recall       F1      MCC   ROC-AUC  Support+
  ---------------------------------------------------------------------------
  Benign                   0.9878     0.9996   0.9936   0.9902    0.9997    15,734
  Brute_Force              1.0000     0.9965   0.9982   0.9982    1.0000       571
  DDoS                     1.0000     0.9797   0.9897   0.9882    0.9996     6,097
  DoS                      0.9951     0.9890   0.9920   0.9907    0.9998     6,362
  MITM/Spoofing            0.9989     0.9775   0.9881   0.9874    1.0000     2,750
  Malware/Mirai            0.9993     0.9816   0.9904   0.9898    0.9999     2,773
  Recon                    0.9993     0.9772   0.9881   0.9849    0.9997    10,096
  Web_Based                1.0000     0.9968   0.9984   0.9984    1.0000       936
  ---------------------------------------------------------------------------
  Macro Avg                                    0.9923   0.9910

  [XGBoost]
  K

####**10.4 Analisis Kelas Sulit - Binary**

In [ ]:
print(f"\n  {'Kelas':<20} {'RF':>8} {'XGB':>8} {'LGB':>8} {'DNN':>8} {'Avg F1':>8}")
print(f"  {'-'*65}")

difficult_bin = []
for class_name in label_list:
    f1_vals = [binary_results[m][class_name]['f1'] for m in model_names_bin]
    avg_f1  = np.mean(f1_vals)
    row     = f"  {class_name:<20}"
    for v in f1_vals:
        row += f" {v:>8.4f}"
    row += f" {avg_f1:>8.4f}"
    print(row)
    if avg_f1 < 0.7:
        difficult_bin.append((class_name, avg_f1))

if difficult_bin:
    print(f"\n  Kelas dengan avg F1 < 0.7:")
    for name, avg in sorted(difficult_bin, key=lambda x: x[1]):
        print(f"    {name:<20} : {avg:.4f}")
else:
    print(f"\n  Semua kelas memiliki avg F1 >= 0.7.")


  Kelas                      RF      XGB      LGB      DNN   Avg F1
  -----------------------------------------------------------------
  Benign                 0.9936   0.9949   0.9951   0.9903   0.9935
  Brute_Force            0.9982   0.9991   0.9974   0.9082   0.9757
  DDoS                   0.9897   0.9908   0.9872   0.9793   0.9868
  DoS                    0.9920   0.9955   0.9921   0.9712   0.9877
  MITM/Spoofing          0.9881   0.9953   0.9927   0.9475   0.9809
  Malware/Mirai          0.9904   0.9938   0.9917   0.9727   0.9871
  Recon                  0.9881   0.9918   0.9897   0.9820   0.9879
  Web_Based              0.9984   0.9984   0.9984   0.9968   0.9980

  Semua kelas memiliki avg F1 >= 0.7.


####**10.5 Simpan Laporan Evaluasi Binary Final**

In [ ]:
best_per_class_bin = {}
for class_name in label_list:
    best_model = max(model_names_bin,
                     key=lambda m: binary_results[m][class_name]['f1'])
    best_per_class_bin[class_name] = {
        'best_model': best_model,
        'f1'        : binary_results[best_model][class_name]['f1'],
        'mcc'       : binary_results[best_model][class_name]['mcc'],
    }

eval_report_bin = {
    'dataset'             : 'CIC IIoT 2025',
    'pendekatan'          : 'Binary One vs Rest (OvR)',
    'n_test_samples'      : int(len(y_test)),
    'n_classes'           : N_CLASSES,
    'label_list'          : label_list,
    'model_results'       : binary_results,
    'best_per_class'      : best_per_class_bin,
    'difficult_classes'   : [
        {'kelas': name, 'avg_f1': round(avg, 4)}
        for name, avg in difficult_bin
    ] if difficult_bin else [],
    'macro_avg_per_model' : {
        m: {
            'macro_f1'  : round(np.mean([binary_results[m][c]['f1']  for c in label_list]), 4),
            'macro_mcc' : round(np.mean([binary_results[m][c]['mcc'] for c in label_list]), 4),
            'macro_roc' : round(np.mean([binary_results[m][c].get('roc_auc', 0) or 0
                                         for c in label_list]), 4),
        }
        for m in model_names_bin
    },
    'has_baseline_comparison': has_baseline_bin,
}

report_path_bin = os.path.join(BINARY_RESULTS_DIR_IIOT,
                               'langkah10_binary_evaluation_report.json')
with open(report_path_bin, 'w') as f:
    json.dump(eval_report_bin, f, indent=4)
print(f"  Laporan JSON tersimpan: {report_path_bin}")

# Simpan full CSV
rows_bin = []
for m in model_names_bin:
    for class_name in label_list:
        res = binary_results[m][class_name]
        rows_bin.append({
            'Model'      : m,
            'Kelas'      : class_name,
            'Precision'  : res['precision'],
            'Recall'     : res['recall'],
            'F1'         : res['f1'],
            'MCC'        : res['mcc'],
            'ROC-AUC'    : res.get('roc_auc', None),
            'Accuracy'   : res['accuracy'],
            'Support Pos': res['support_pos'],
            'Support Neg': res['support_neg'],
            'Train Time(s)': res.get('train_time_s', None),
        })

df_bin_full = pd.DataFrame(rows_bin)
full_csv_bin = os.path.join(BINARY_RESULTS_DIR_IIOT,
                            'langkah10_binary_full_metrics.csv')
df_bin_full.to_csv(full_csv_bin, index=False)
print(f"  Full metrics CSV tersimpan: {full_csv_bin}")

  Laporan JSON tersimpan: /content/drive/My Drive/Framework/CICIIoT2025/Binary/Results/langkah10_binary_evaluation_report.json
  Full metrics CSV tersimpan: /content/drive/My Drive/Framework/CICIIoT2025/Binary/Results/langkah10_binary_full_metrics.csv
